In [87]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [88]:
df = pd.read_csv("data/data.csv")
df.head()

,Age,BMI,Sex,Height,Weight,Length_of_Stay,Management,Severity,Diagnosis_Presumptive,Diagnosis,...,Abscess_Location,Pathological_Lymph_Nodes,Lymph_Nodes_Location,Bowel_Wall_Thickening,Conglomerate_of_Bowel_Loops,Ileus,Coprostasis,Meteorism,Enteritis,Gynecological_Findings
0,12.68,16.9,female,148.0,37.0,3.0,conservative,uncomplicated,appendicitis,appendicitis,...,NaN,yes,reUB,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,14.10,31.9,male,147.0,69.5,2.0,conservative,uncomplicated,appendicitis,no appendicitis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,NaN
2,14.14,23.3,female,163.0,62.0,4.0,conservative,uncomplicated,appendicitis,no appendicitis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,yes,NaN
3,16.37,20.6,female,165.0,56.0,3.0,conservative,uncomplicated,appendicitis,no appendicitis,...,NaN,yes,reUB,NaN,NaN,NaN,NaN,NaN,yes,NaN
4,11.08,16.9,female,163.0,45.0,3.0,conservative,uncomplicated,appendicitis,appendicitis,...,NaN,yes,reUB,NaN,NaN,NaN,NaN,NaN,yes,NaN


In [89]:
df.shape

(782, 58)

In [90]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 782 entries, 0 to 781
Data columns (total 58 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Age                               781 non-null    float64
 1   BMI                               755 non-null    float64
 2   Sex                               780 non-null    str    
 3   Height                            756 non-null    float64
 4   Weight                            779 non-null    float64
 5   Length_of_Stay                    778 non-null    float64
 6   Management                        781 non-null    str    
 7   Severity                          781 non-null    str    
 8   Diagnosis_Presumptive             780 non-null    str    
 9   Diagnosis                         780 non-null    str    
 10  Alvarado_Score                    730 non-null    float64
 11  Paedriatic_Appendicitis_Score     730 non-null    float64
 12  Appendix_on_US     

In [91]:
df["Diagnosis"].value_counts()

Diagnosis
appendicitis       463
no appendicitis    317
Name: count, dtype: int64

In [92]:
df.isnull().sum().sort_values(ascending=False).head(20)

Abscess_Location               769
Gynecological_Findings         756
Conglomerate_of_Bowel_Loops    739
Segmented_Neutrophils          728
Ileus                          722
Perfusion                      719
Enteritis                      716
Appendicolith                  713
Coprostasis                    711
Perforation                    701
Appendicular_Abscess           697
Bowel_Wall_Thickening          683
Lymph_Nodes_Location           661
Target_Sign                    644
Meteorism                      642
Pathological_Lymph_Nodes       579
Appendix_Wall_Layers           564
Surrounding_Tissue_Reaction    530
Appendix_Diameter              284
RBC_in_Urine                   206
dtype: int64

## Observations so far
- Dataset has 782 rows and 58 columns
- Target is Diagnosis: 463 appendicitis, 317 no appendicitis (fairly balanced)
- 2 rows have no diagnosis
- Most of the heavily missing columns are ultrasound findings, so using only clinical and lab features
- Segmented_Neutrophils is a lab column but almost fully empty

In [93]:
df = df.dropna(subset=["Diagnosis"])
df.shape

(780, 58)

In [94]:
leak_cols = ["Management", "Severity", "Diagnosis_Presumptive", "Length_of_Stay"]
df = df.drop(columns=leak_cols)
df.shape

(780, 54)

In [95]:
us_cols = [
    "Abscess_Location", "Gynecological_Findings", "Conglomerate_of_Bowel_Loops",
    "Ileus", "Perfusion", "Enteritis", "Appendicolith", "Coprostasis",
    "Perforation", "Appendicular_Abscess", "Bowel_Wall_Thickening",
    "Lymph_Nodes_Location", "Target_Sign", "Meteorism",
    "Pathological_Lymph_Nodes", "Appendix_Wall_Layers",
    "Surrounding_Tissue_Reaction", "Appendix_Diameter", "Free_Fluids",
    "US_Number", "Appendix_on_US", "US_Performed",
]
df = df.drop(columns=us_cols)
df.shape

(780, 32)

## Cleaning so far
- Dropped 2 rows with no diagnosis (780 rows left)
- Dropped Management, Severity, Diagnosis_Presumptive because they leak the answer
- Dropped all 22 ultrasound columns, 33 columns left

In [96]:
df = df.drop(columns=["Segmented_Neutrophils"])
df.shape

(780, 31)

In [97]:
df = df.drop(columns=["Alvarado_Score", "Paedriatic_Appendicitis_Score"])
df.shape

(780, 29)

In [98]:
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

RBC_in_Urine                        204
Ketones_in_Urine                    198
WBC_in_Urine                        197
Ipsilateral_Rebound_Tenderness      161
Neutrophil_Percentage               101
Neutrophilia                         48
Psoas_Sign                           35
Dysuria                              27
BMI                                  26
Height                               25
RDW                                  24
RBC_Count                            16
Thrombocyte_Count                    16
Hemoglobin                           16
Stool                                15
Coughing_Pain                        14
Contralateral_Rebound_Tenderness     13
CRP                                   9
Loss_of_Appetite                      8
Peritonitis                           7
Migratory_Pain                        7
Nausea                                6
Lower_Right_Abd_Pain                  6
Body_Temperature                      5
WBC_Count                             4


In [99]:
(204/780) * 100

26.153846153846157

In [100]:
X = df.drop(columns=["Diagnosis"])
y = df["Diagnosis"]
print(X.shape, y.shape)
y.value_counts()

(780, 28) (780,)


Diagnosis
appendicitis       463
no appendicitis    317
Name: count, dtype: int64

In [101]:
from sklearn.model_selection import train_test_split
y = y.map({
    "appendicitis": 1,
    "no appendicitis": 0
})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(624, 28) (156, 28)
Diagnosis
1    370
0    254
Name: count, dtype: int64
Diagnosis
1    93
0    63
Name: count, dtype: int64


## Cleaning and Split
- Dropped Segmented_Neutrophils (almost fully empty)
- Dropped Alvarado_Score and Paedriatic_Appendicitis_Score (calculated from the same features)
- Dropped Length_of_Stay (only known after treatment, so it leaks the outcome)
- X has 28 features, y is 1 for appendicitis and 0 for no appendicitis